# PaperMind — ByT5 OCR Correction (Full Pipeline)

**One notebook does everything:**
1. Generate `(noisy_trocr_output, clean_text)` pairs using the real TrOCR model on this GPU
2. Fine-tune `google/byt5-small` on those pairs
3. Evaluate and save the checkpoint to your Google Drive

**Before running:**
- Runtime → Change runtime type → **T4 GPU** (required)
- Estimated total time: ~3–4 hours for 30k pairs

**Config cell (Cell 2) is the only thing you need to touch.**

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────
!pip install -q transformers==4.44.* datasets evaluate sacrebleu sentencepiece accelerate Pillow
print('Deps installed.')

In [ ]:
# ── Cell 2: CONFIG — only cell you need to change ─────────────────────────

# How many (noisy, clean) training pairs to generate.
# 30_000 → ~1.5h generation + ~2h training = ~3.5h total on T4
# 10_000 → ~30min generation + ~45min training = ~1.25h total (good for a test run)
N_PAIRS = 30_000

# TrOCR batch size during dataset generation.
# 32 works on T4 16GB. Lower to 16 if you get OOM.
TROCR_BATCH = 32

# ByT5 fine-tuning batch size per device.
BYT5_TRAIN_BATCH = 32
BYT5_EVAL_BATCH  = 64

# Training epochs
EPOCHS = 5

# Drive output paths
DRIVE_DATA_DIR   = '/content/drive/MyDrive/papermind/correction'
DRIVE_CKPT_DIR   = '/content/drive/MyDrive/papermind/byt5-correction-checkpoint'

# Random seed
SEED = 42

print(f'Config: N_PAIRS={N_PAIRS:,}  EPOCHS={EPOCHS}  TROCR_BATCH={TROCR_BATCH}')

In [ ]:
# ── Cell 3: Mount Drive + verify GPU ─────────────────────────────────────
import os, torch
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU detected — change runtime to T4!'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Drive: {DRIVE_DATA_DIR}')

In [ ]:
# ── Cell 4: Academic corpus ───────────────────────────────────────────────
# Text that covers PaperMind's exam domains: OOP, SE, AI/ML, Cybersecurity.
# Includes question markers (1), Q2., Ans 3:) and option letters ((a)(b)(c))
# because those are the highest-error-rate patterns in TrOCR.
import random, math
random.seed(SEED)

_MARKERS = ['1)', '2)', '3)', '4)', '5)', '6)',
            'Q1.', 'Q2.', 'Q3.', 'Q4.',
            'Ans 1:', 'Ans 2:', 'Ans 3:']

_OOP = [
    'Encapsulation bundles data and methods into a single class.',
    'Inheritance allows a subclass to reuse the behaviour of a superclass.',
    'Polymorphism enables objects of different classes to respond to the same method call.',
    'Abstraction hides implementation details and exposes only the interface.',
    'The Liskov Substitution Principle: objects of a subclass should replace the superclass.',
    'A constructor is invoked automatically when an object is created.',
    'The final keyword prevents a class from being subclassed.',
    'Method overloading: same name, different parameter lists.',
    'Method overriding: subclass provides a different implementation.',
    'An interface defines a contract that implementing classes must fulfil.',
    'Abstract classes cannot be instantiated directly.',
    'The singleton pattern ensures only one instance of a class exists.',
    'Composition is preferred over inheritance to reduce coupling.',
    'Access modifiers: public, private, protected, and default.',
    'Static members belong to the class rather than to any instance.',
    'A stack uses Last In First Out ordering.',
    'A queue uses First In First Out ordering.',
    'Binary search runs in O(log n) time on a sorted array.',
    'Quicksort has an average time complexity of O(n log n).',
    'The factory method pattern defines an interface for creating objects.',
    'The observer pattern defines a one-to-many dependency between objects.',
    'Dependency injection reduces tight coupling between components.',
    'The decorator pattern attaches additional responsibilities dynamically.',
    'Garbage collection automatically reclaims memory from unused objects.',
    'Synchronisation prevents race conditions in multithreaded programs.',
]

_SE = [
    'Agile methodology emphasises iterative development and customer collaboration.',
    'The waterfall model is a linear sequential approach to software development.',
    'Regression testing verifies that new changes have not broken existing functionality.',
    'Continuous integration merges code changes into a shared repository frequently.',
    'SOLID: Single responsibility, Open-closed, Liskov, Interface segregation, Dependency inversion.',
    'A sprint in Scrum is a time-boxed iteration typically lasting two weeks.',
    'Test-driven development writes tests before writing the implementation code.',
    'Static analysis tools detect potential bugs without executing the program.',
    'McCabe cyclomatic complexity measures the number of linearly independent paths.',
    'Technical debt accumulates when shortcuts are taken instead of proper solutions.',
]

_AI = [
    'Supervised learning uses labelled training data to map inputs to outputs.',
    'Gradient descent minimises a loss function by iteratively updating parameters.',
    'Overfitting occurs when a model learns training data noise rather than the pattern.',
    'A convolutional neural network is well suited for image recognition tasks.',
    'The attention mechanism allows the model to focus on relevant input positions.',
    'Precision measures the fraction of predicted positives that are actually positive.',
    'Recall measures the fraction of actual positives correctly predicted.',
    'The F1-score is the harmonic mean of precision and recall.',
    'K-means clustering partitions data into k clusters by minimising intra-cluster distance.',
    'Transfer learning reuses a model trained on one task for a different related task.',
]

_CYBER = [
    'Symmetric encryption uses the same key for encryption and decryption.',
    'Asymmetric encryption uses a public key to encrypt and a private key to decrypt.',
    'SQL injection inserts malicious input into a SQL query.',
    'Cross-site scripting injects malicious scripts into web pages viewed by others.',
    'Two-factor authentication requires two forms of identification.',
    'Hashing converts data into a fixed-length string that cannot be reversed.',
    'The principle of least privilege grants only the minimum access required.',
]

_SHORT = [
    'True', 'False',
    '(a) Encapsulation', '(b) Inheritance', '(c) Polymorphism', '(d) Abstraction',
    '(a) True', '(b) False',
    '(a) O(1)', '(b) O(log n)', '(c) O(n)', '(d) O(n log n)', '(e) O(n^2)',
    '(a) Stack', '(b) Queue', '(c) Heap', '(d) Tree',
    'True, because encapsulation hides internal state.',
    'False, because interfaces cannot have instance variables.',
    'True, polymorphism allows runtime method dispatch.',
    'False, an abstract class can have concrete methods.',
]

def build_corpus(n):
    base = _OOP*5 + _SE*3 + _AI*3 + _CYBER*2 + _SHORT*10
    # Question marker + sentence combos are the highest-error patterns
    with_markers = []
    for m in _MARKERS:
        for s in _OOP + _SE + _AI:
            with_markers.append(f'{m} {s}')
        for s in _SHORT:
            with_markers.append(f'{m} {s}')
    pool = base + with_markers * 3
    random.shuffle(pool)
    times = math.ceil(n / len(pool))
    out = []
    for _ in range(times):
        t = pool[:]; random.shuffle(t); out.extend(t)
    return out[:n]

corpus = build_corpus(N_PAIRS * 4)  # over-generate; we filter misses
print(f'Corpus size: {len(corpus):,} sentences')
print('Sample:', corpus[:3])

In [ ]:
# ── Cell 5: Image renderer ────────────────────────────────────────────────
# Renders each sentence as a synthetic handwriting-style line image.
# Uses fonts available in Colab (DejaVu, Liberation, Courier).
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter

FONT_PATHS = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf',
]
FONT_SIZES = [24, 28, 32, 36]

def _load_font(size):
    for p in FONT_PATHS:
        if os.path.exists(p):
            try: return ImageFont.truetype(p, size)
            except: pass
    return ImageFont.load_default()

# Pre-load font objects (avoid repeated disk reads)
_fonts = {(p, s): _load_font(s) for s in FONT_SIZES for p in FONT_PATHS if os.path.exists(p)}
print(f'Loaded {len(_fonts)} font variants')

def render_line(text, font_size=None, noise_level=0.0):
    """Render text as a greyscale handwriting-like line image.
    noise_level in [0, 1]: 0=clean, 1=heavy degradation."""
    if font_size is None:
        font_size = random.choice(FONT_SIZES)
    font = _load_font(font_size)

    # Measure
    dummy = Image.new('RGB', (1, 1))
    draw = ImageDraw.Draw(dummy)
    bbox = draw.textbbox((0, 0), text, font=font)
    tw, th = bbox[2]-bbox[0], bbox[3]-bbox[1]
    px, py = max(10, int(tw*0.06)), max(6, int(th*0.20))
    W, H = tw + 2*px, th + 2*py

    # Draw
    img = Image.new('RGB', (W, H), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    ink = (random.randint(0,25), random.randint(0,25), random.randint(0,40))
    draw.text((px, py), text, font=font, fill=ink)

    # Noise
    if noise_level > 0:
        arr = np.asarray(img, dtype=np.float32)
        if noise_level > 0.2:
            arr += np.random.normal(0, noise_level * 22, arr.shape)
            arr = np.clip(arr, 0, 255)
        if noise_level > 0.4:
            mask = np.random.random(arr.shape[:2]) < noise_level * 0.025
            arr[mask] = np.random.choice([0, 255])
        img = Image.fromarray(arr.astype(np.uint8))
        if noise_level > 0.3:
            angle = random.uniform(-noise_level*3.5, noise_level*3.5)
            img = img.rotate(angle, fillcolor=(255,255,255), expand=False)
        if noise_level > 0.5:
            img = img.filter(ImageFilter.GaussianBlur(radius=noise_level*1.2))
    return img.convert('RGB')

# Quick render test
test_img = render_line('1) (b) Encapsulation hides state.', noise_level=0.4)
print(f'Render test OK — image size: {test_img.size}')

In [ ]:
# ── Cell 6: Load TrOCR ───────────────────────────────────────────────────
# microsoft/trocr-base-handwritten — the SAME model PaperMind uses in production.
# This means the error patterns we capture are exactly the errors we need to fix.
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

TROCR_MODEL = 'microsoft/trocr-base-handwritten'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading TrOCR ({TROCR_MODEL})…')
trocr_proc  = TrOCRProcessor.from_pretrained(TROCR_MODEL)
trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL).eval().to(device)
print('TrOCR ready on', device)

@torch.inference_mode()
def trocr_batch(images):
    """Run TrOCR on a list of PIL images. Returns list of (text, confidence)."""
    pixels = trocr_proc(images=images, return_tensors='pt').pixel_values.to(device)
    out = trocr_model.generate(
        pixels, max_new_tokens=64, num_beams=1,
        output_scores=True, return_dict_in_generate=True
    )
    logprobs = trocr_model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
    pad = trocr_proc.tokenizer.pad_token_id
    tokens = out.sequences[:, 1:]
    texts  = trocr_proc.batch_decode(out.sequences, skip_special_tokens=True)
    results = []
    for text, lp, tok in zip(texts, logprobs, tokens):
        mask = tok != pad
        mean_lp = float(lp[mask].mean()) if mask.any() else -5.0
        results.append((text.strip(), math.exp(mean_lp)))
    return results

In [ ]:
# ── Cell 7: GENERATE DATASET (TrOCR on GPU) ──────────────────────────────
# This is the key cell. We:
#  1. Render each corpus sentence as a line image (Pillow)
#  2. Run TrOCR on it to get the noisy recognition output
#  3. Save the (noisy, clean) pair if there's any edit distance
#  4. Keep a controlled fraction of perfect pairs (so model doesn't over-correct)
import csv, time
from pathlib import Path

_NOISE_LEVELS = [0.0, 0.0, 0.12, 0.25, 0.35, 0.50, 0.65]
CLEAN_FRACTION = 0.15   # fraction of ed=0 pairs to keep

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for ca in a:
        curr = [prev[0] + 1]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j]+1, curr[-1]+1, prev[j-1]+(ca!=cb)))
        prev = curr
    return prev[-1]

pairs = []
corpus_iter = iter(corpus)
t0 = time.time()

print(f'Generating {N_PAIRS:,} pairs (TrOCR batch={TROCR_BATCH})…')
print('Each dot = 1 batch processed')

while len(pairs) < N_PAIRS:
    batch_texts, batch_imgs, batch_noise = [], [], []

    for _ in range(TROCR_BATCH):
        try:
            text = next(corpus_iter)
        except StopIteration:
            corpus_iter = iter(corpus)
            text = next(corpus_iter)
        nl = random.choice(_NOISE_LEVELS)
        img = render_line(text, font_size=random.choice(FONT_SIZES), noise_level=nl)
        batch_texts.append(text)
        batch_imgs.append(img)
        batch_noise.append(nl)

    results = trocr_batch(batch_imgs)

    for clean, (noisy, conf), nl in zip(batch_texts, results, batch_noise):
        ed = levenshtein(noisy, clean)
        if ed == 0 and random.random() > CLEAN_FRACTION:
            continue
        pairs.append({'input': noisy, 'target': clean,
                      'edit_distance': ed, 'trocr_conf': round(conf,4),
                      'noise_level': round(nl,2)})

    n = len(pairs)
    if (n // 500) > ((n - len(results)) // 500) or n >= N_PAIRS:
        elapsed = time.time() - t0
        rate = n / elapsed if elapsed > 0 else 0
        eta  = (N_PAIRS - n) / rate if rate > 0 else 0
        print(f'  pairs={n:>6,}/{N_PAIRS:,}  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m')

    if len(pairs) >= N_PAIRS:
        break

pairs = pairs[:N_PAIRS]
random.shuffle(pairs)

# ── Stats ──────────────────────────────────────────────────────────────────
total    = len(pairs)
perfect  = sum(1 for p in pairs if p['edit_distance'] == 0)
avg_ed   = sum(p['edit_distance'] for p in pairs) / total
avg_conf = sum(p['trocr_conf']    for p in pairs) / total

print(f'\n=== Dataset stats ===')
print(f'  Total pairs   : {total:,}')
print(f'  Perfect (ed=0): {perfect:,}  ({100*perfect/total:.1f}%)')
print(f'  Avg edit dist : {avg_ed:.2f}')
print(f'  Avg TrOCR conf: {avg_conf:.3f}')
print(f'  Generation time: {(time.time()-t0)/60:.1f} min')
print()

# ── Sample errors ─────────────────────────────────────────────────────────
print('=== Sample error pairs ===')
shown = 0
for p in pairs:
    if p['edit_distance'] > 0:
        print(f'  IN : {p["input"]!r}')
        print(f'  OUT: {p["target"]!r}')
        print(f'  ED={p["edit_distance"]}  conf={p["trocr_conf"]}')
        print()
        shown += 1
    if shown >= 8: break

In [ ]:
# ── Cell 8: Save dataset CSVs ─────────────────────────────────────────────
import csv
from pathlib import Path

n      = len(pairs)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
splits  = {
    'train': pairs[:n_train],
    'val':   pairs[n_train : n_train + n_val],
    'test':  pairs[n_train + n_val :],
}
fields = ['input', 'target', 'edit_distance', 'trocr_conf', 'noise_level']

for name, rows in splits.items():
    path = Path(DRIVE_DATA_DIR) / f'{name}.csv'
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Saved {path} ({len(rows):,} rows)')

print('\nDataset saved to Drive.')

In [ ]:
# ── Cell 9: Free TrOCR VRAM before ByT5 training ─────────────────────────
# ByT5-small + TrOCR together may exceed T4 VRAM.
# We're done generating, so delete TrOCR from GPU memory.
del trocr_model
del trocr_proc
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'VRAM free: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB reserved')
print('TrOCR unloaded. Ready for ByT5 fine-tuning.')

In [ ]:
# ── Cell 10: Build HuggingFace Dataset from pairs in memory ───────────────
import pandas as pd
from datasets import Dataset, DatasetDict

def to_hf(rows):
    return Dataset.from_dict({
        'input':  [r['input']  for r in rows],
        'target': [r['target'] for r in rows],
    })

raw_datasets = DatasetDict({
    'train': to_hf(splits['train']),
    'val':   to_hf(splits['val']),
    'test':  to_hf(splits['test']),
})
print(raw_datasets)

In [ ]:
# ── Cell 11: ByT5 tokeniser ───────────────────────────────────────────────
# ByT5 encodes text as raw UTF-8 bytes.
# No vocabulary — every byte 0x00-0xFF is its own token.
# This is perfect for character-level OCR errors.
from transformers import AutoTokenizer

BMODEL     = 'google/byt5-small'
MAX_LEN    = 256    # bytes — one OCR line is rarely > 150 chars

byt5_tok = AutoTokenizer.from_pretrained(BMODEL)
print(f'Tokeniser: {type(byt5_tok).__name__}  vocab_size={byt5_tok.vocab_size}')

def preprocess(examples):
    inp = byt5_tok(
        examples['input'],
        max_length=MAX_LEN, padding='max_length', truncation=True
    )
    with byt5_tok.as_target_tokenizer():
        lbl = byt5_tok(
            examples['target'],
            max_length=MAX_LEN, padding='max_length', truncation=True
        )
    # -100 masks padding so it's excluded from the cross-entropy loss
    lbl['input_ids'] = [
        [(t if t != byt5_tok.pad_token_id else -100) for t in seq]
        for seq in lbl['input_ids']
    ]
    inp['labels'] = lbl['input_ids']
    return inp

tokenized = raw_datasets.map(
    preprocess, batched=True, remove_columns=['input', 'target'], desc='Tokenising'
)
print(tokenized)

In [ ]:
# ── Cell 12: Load ByT5-small ──────────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM

byt5 = AutoModelForSeq2SeqLM.from_pretrained(BMODEL)
n_params = sum(p.numel() for p in byt5.parameters())
print(f'Model: {BMODEL}  |  {n_params/1e6:.0f}M parameters')
print(f'VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Cell 13: Metrics ──────────────────────────────────────────────────────
import numpy as np

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    if preds.ndim == 3: preds = np.argmax(preds, axis=-1)

    dec_preds  = byt5_tok.batch_decode(preds, skip_special_tokens=True)
    labels_np  = np.where(labels != -100, labels, byt5_tok.pad_token_id)
    dec_labels = byt5_tok.batch_decode(labels_np, skip_special_tokens=True)

    exact = sum(p.strip()==l.strip() for p,l in zip(dec_preds,dec_labels)) / len(dec_preds)

    total_ed, total_len = 0, 0
    for p, l in zip(dec_preds, dec_labels):
        total_ed  += levenshtein(p.strip(), l.strip())
        total_len += max(len(l.strip()), 1)

    return {'exact_match': round(exact, 4), 'cer': round(total_ed/total_len, 4)}

print('Metrics: exact_match ↑  CER ↓  (target: CER < 0.05)')

In [ ]:
# ── Cell 14: Trainer setup ────────────────────────────────────────────────
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

args = Seq2SeqTrainingArguments(
    output_dir=DRIVE_CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BYT5_TRAIN_BATCH,
    per_device_eval_batch_size=BYT5_EVAL_BATCH,
    learning_rate=5e-4,
    warmup_steps=200,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,      # lower CER is better
    save_total_limit=2,
    fp16=True,                    # mixed precision on T4
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,
    logging_steps=100,
    report_to='none',
)

collator = DataCollatorForSeq2Seq(byt5_tok, model=byt5, padding=True, pad_to_multiple_of=8)

trainer = Seq2SeqTrainer(
    model=byt5,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    tokenizer=byt5_tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
print(f'Trainer ready. Train={len(tokenized["train"]):,}  Val={len(tokenized["val"]):,}')
print(f'Effective batch size = {BYT5_TRAIN_BATCH * 2} (grad_accum=2)')
steps_per_epoch = len(tokenized['train']) // (BYT5_TRAIN_BATCH * 2)
print(f'Steps/epoch ≈ {steps_per_epoch:,}  |  Total steps ≈ {steps_per_epoch * EPOCHS:,}')

In [ ]:
# ── Cell 15: TRAIN ───────────────────────────────────────────────────────
# Expected: ~1.5–2.5h on T4 for 30k pairs × 5 epochs.
# Best checkpoint (lowest val CER) is auto-loaded at the end.
# Watch the CER column — it should drop from ~0.3 → < 0.05 by epoch 3.
train_t0 = time.time()
result = trainer.train()
print(f'Training done in {(time.time()-train_t0)/60:.1f} min')
print(result.metrics)

In [ ]:
# ── Cell 16: Save best model to Drive ────────────────────────────────────
trainer.save_model(DRIVE_CKPT_DIR)
byt5_tok.save_pretrained(DRIVE_CKPT_DIR)
print(f'Checkpoint saved → {DRIVE_CKPT_DIR}')
print('Files:', sorted(os.listdir(DRIVE_CKPT_DIR)))

In [ ]:
# ── Cell 17: Test set evaluation ──────────────────────────────────────────
test_res = trainer.evaluate(tokenized['test'])
print('=== Test Results ===')
print(f"  Exact Match : {test_res.get('eval_exact_match', 0):.4f}")
print(f"  CER         : {test_res.get('eval_cer', 1):.4f}")
print(f"  Loss        : {test_res.get('eval_loss', 0):.4f}")
print()
cer = test_res.get('eval_cer', 1)
if cer < 0.05:
    print('✓ CER < 0.05 — model is ready for production use.')
elif cer < 0.10:
    print('~ CER < 0.10 — usable but consider more epochs or more data.')
else:
    print('✗ CER >= 0.10 — consider increasing N_PAIRS or EPOCHS.')

In [ ]:
# ── Cell 18: Qualitative check — actual PaperMind TrOCR errors ───────────
# These are the real errors from storage_demo/answer_sheets/7/ocr.json
real_errors = [
    ('19 ( 0b ) .',                                  '1) (b).'),
    ('2 ) False',                                    '2) False'),
    ('3 ) Liskov . principle says objects of a',     '3) Liskov principle says objects of a'),
    ('program . Square extending . i Rectangle',     'program. Square extending Rectangle'),
    ('4 ) Encapsulation bundles data and methods',   '4) Encapsulation bundles data and methods'),
    ('using private access . It protects the data',  'using private access. It protects the data'),
    ('lnheritance aIIows a subclass',                'Inheritance allows a subclass'),
    ('0bject-oriented prograrnming',                 'Object-oriented programming'),
    ('Q1 . (b)',                                     'Q1. (b)'),
    ('rn ethod overriding',                          'method overriding'),
]

byt5.eval()
dev = next(byt5.parameters()).device
print('=== Qualitative evaluation — real PaperMind OCR errors ===')
exact, improved, missed = 0, 0, 0

for noisy, expected in real_errors:
    inp = byt5_tok(noisy, return_tensors='pt', max_length=256, truncation=True)
    inp = {k: v.to(dev) for k, v in inp.items()}
    with torch.no_grad():
        out = byt5.generate(**inp, max_new_tokens=256, num_beams=4)
    pred    = byt5_tok.decode(out[0], skip_special_tokens=True).strip()
    ed_in   = levenshtein(noisy, expected)
    ed_out  = levenshtein(pred, expected)

    if pred == expected:
        tag = '✓ EXACT  '; exact += 1
    elif ed_out < ed_in:
        tag = '~ IMPROVED'; improved += 1
    else:
        tag = '✗ MISS   '; missed += 1

    print(f'{tag}  IN : {noisy!r}')
    if pred != expected:
        print(f'          OUT: {pred!r}')
        print(f'          EXP: {expected!r}')
    print(f'          ED  {ed_in} → {ed_out}')
    print()

print(f'Summary: {exact} exact  {improved} improved  {missed} missed  /  {len(real_errors)} total')

In [ ]:
# ── Cell 19: Integration instructions ────────────────────────────────────
print('=' * 65)
print('DONE — integrate the checkpoint into PaperMind backend')
print('=' * 65)
print()
print('1. In Google Drive, navigate to:')
print(f'   {DRIVE_CKPT_DIR}')
print('   Right-click → Download → will download as a zip.')
print()
print('2. Extract the zip on your Windows machine to:')
print('   C:\\PaperMind\\backend\\models\\byt5-correction\\')
print('   The folder must contain: config.json + tokenizer.json + model.safetensors')
print()
print('3. Open C:\\PaperMind\\backend\\.env and add/update:')
print('   CORRECTION_MODEL_PATH=./models/byt5-correction')
print()
print('4. Restart the backend (Uvicorn). Look for this log line:')
print('   seq2seq:./models/byt5-correction')
print()
print('5. Upload a real answer sheet via the UI and check the')
print('   corrected.json in storage_demo/answer_sheets/<id>/')
print('   The Q1 error ("19 ( 0b ) .") should now read "1) (b)."')
print()
print(f'Final CER  : {test_res.get("eval_cer", "?")}  (target < 0.05)')
print(f'Final EM   : {test_res.get("eval_exact_match", "?")}')